In [1]:
import pandas as pd
import gc

from collections import defaultdict
from surprise import KNNBasic, Dataset, Reader
from tqdm import tqdm

In [2]:
train = pd.read_csv('../dataset/train_A.tsv', sep='\t')

In [3]:
test = pd.read_csv('../dataset/test.tsv', sep='\t')
test_A = test[test['user_id'].str.endswith('_A')]

In [4]:
# 関連度スコアの計算
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

In [5]:
# 関連度スコアの設定
train['relevance'] = train.apply(compute_relevance, axis=1)

In [6]:
# 評価(スコア)範囲の設定
reader = Reader(rating_scale=(0, 3))

In [7]:
# データの準備
data = Dataset.load_from_df(train[['user_id', 'product_id', 'relevance']], reader)

In [8]:
# 各アイテムの評価数をカウント
trainset = data.build_full_trainset()

item_ratings_count = defaultdict(int)
for user, item, rating in trainset.all_ratings():
    item_ratings_count[item] += 1

In [9]:
# 評価数が少ないアイテムをフィルタリング
threshold = 2  # 共通評価数が2以上のアイテムペアを対象にする
filtered_items = [item for item, count in item_ratings_count.items() if count >= threshold]

In [10]:
# 訓練データをフィルタリング
filtered_data = train[train['product_id'].isin(filtered_items)]
reader = Reader(rating_scale=(0, 3))
filtered_data = Dataset.load_from_df(filtered_data[['user_id', 'product_id', 'score']], reader)
trainset = filtered_data.build_full_trainset()

KeyError: "['score'] not in index"

In [ ]:
# 商品一覧の取得
unique_products = train['product_id'].drop_duplicates()

In [ ]:
# 学習
sim_options = {
    'name': 'cosine',
    'user_based': False  # アイテムベースに変更
}
algo = KNNBasic(sim_options=sim_options)
algo.fit(trainset)

In [ ]:
# 予測
batch_predictions = []

for user in tqdm(test_A['user_id']):
    user_preds = []

    for product in unique_products:
        pred = algo.predict(user, product)
        user_preds.append({
            'user_id': user,
            'product_id': product,
            'score': pred.est
        })

    # 上位22だけ保存
    top_preds = sorted(user_preds, key=lambda x: x['score'], reverse=True)[:22]
    batch_predictions.extend(top_preds)

    # メモリ解放
    del user_preds
    gc.collect()

# 最終的にDataFrameに変換
predictions_df = pd.DataFrame(batch_predictions)

In [ ]:
# 各ユーザーごとにスコア上位22件を抽出
top_22_per_user = (
    predictions_df
    .sort_values(['user_id', 'score'], ascending=[True, False])
    .groupby('user_id')
    .head(22)
    .reset_index(drop=True)
)

In [ ]:
# 関連度ランクの計算
top_22_per_user['rank'] = (
    top_22_per_user
    .groupby('user_id')
    .cumcount() + 1
)

In [ ]:
pd.DataFrame(top_22_per_user, columns=['user_id', 'product_id', 'rank']).to_csv('../dataset/predict4_A.tsv', sep='\t', index=False)